# Qwen3-4B Multi-Agent Test Environment

## Setup
- Cell 1: pip install + 코드 전체 + 모델 로드 (최초 1회)
- Cell 2: 에이전트 생성 + 프롬프트 정의
- Cell 3+: 실험 (복사해서 쓰면 됨)

---

## API Reference

### Agent 생성
```python
agent = Agent("이름", "system prompt")
```

### 단일 추론
```python
# 기본 (thinking OFF, 256 tokens)
r = agent.say("질문")

# thinking ON + 토큰 늘리기
r = agent.say("질문", max_tokens=2048, thinking=True)

# 결과 접근
r["response"]   # 응답 텍스트
r["tokens"]     # 생성 토큰 수
r["time"]       # 소요 시간(초)
```

### 파라미터
| 파라미터 | 기본값 | 설명 |
|----------|--------|------|
| `max_tokens` | 256 | 생성 최대 토큰. thinking=True면 2048 권장 |
| `thinking` | False | True: Qwen3 내부 추론(`<think>`) 활성화. 느리지만 정확도↑ |

### 프롬프트 변경
```python
agent.set_prompt("새 system prompt")
```

### A → B 통신
```python
result = send(a, b, "메시지")
result["sender"]["response"]    # A의 응답
result["receiver"]["response"]  # B의 응답
result["total_tokens"]          # 총 토큰
```

### A → B → C 체인
```python
result = chain([a, b, c], "메시지")
result["final"]          # 마지막 에이전트 응답
result["total_tokens"]   # 총 토큰
result["steps"]          # 각 단계별 결과 리스트
```

### ask() 헬퍼 (ABCD 문제용)
```python
r = ask(agent, MATH_PROMPT, "질문텍스트", max_tokens=512)
r["answer"]     # 추출된 답 (A/B/C/D 또는 N/A)
r["response"]   # 전체 응답
r["tokens"]     # 토큰 수
r["time"]       # 소요 시간
```

### 유틸리티
```python
extract_number("답은 42입니다")   # → 42.0
grade(got, expected)              # 채점 (10% 오차 허용)
```

In [ ]:
# ============================================================
# Cell 1: 모델 로드 (최초 1회)
# ============================================================
!pip install -q torch transformers accelerate datasets

import torch
import time
import re

# Global model/tokenizer
_model = None
_tokenizer = None
_device = None

def load_model(model_id: str = "Qwen/Qwen3-4B"):
    global _model, _tokenizer, _device
    from transformers import AutoModelForCausalLM, AutoTokenizer
    _device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if _device == "cuda" else torch.float32
    print(f"Device: {_device}")
    if _device == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Loading {model_id}...")
    t0 = time.time()
    _tokenizer = AutoTokenizer.from_pretrained(model_id)
    _model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=dtype,
        device_map="auto" if _device == "cuda" else None,
    )
    if _device == "cpu":
        _model = _model.to(_device)
    params = sum(p.numel() for p in _model.parameters()) / 1e9
    print(f"Loaded in {time.time()-t0:.1f}s ({params:.1f}B params)")

class Agent:
    def __init__(self, name: str, system_prompt: str):
        self.name = name
        self.system_prompt = system_prompt
        self.history = []

    def say(self, message: str, max_tokens: int = 256, thinking: bool = False) -> dict:
        if _model is None:
            raise RuntimeError("load_model()을 먼저 실행하세요.")
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": message},
        ]
        text = _tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=thinking
        )
        inputs = _tokenizer(text, return_tensors="pt").to(_model.device)
        t0 = time.time()
        with torch.no_grad():
            output = _model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
        elapsed = time.time() - t0
        response = _tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        gen_tokens = output.shape[1] - inputs["input_ids"].shape[1]
        result = {"response": response, "tokens": gen_tokens, "time": round(elapsed, 1)}
        self.history.append({"input": message, **result})
        return result

    def set_prompt(self, new_prompt: str):
        self.system_prompt = new_prompt
        return self

    def __repr__(self):
        return f"Agent('{self.name}')"

def send(sender, receiver, message, max_tokens=256, verbose=True):
    s = sender.say(message, max_tokens=max_tokens)
    r = receiver.say(s["response"], max_tokens=max_tokens)
    if verbose:
        print(f"[{sender.name}] {s['tokens']}tok, {s['time']}s")
        print(f"  >> {s['response'][:200]}")
        print(f"[{receiver.name}] {r['tokens']}tok, {r['time']}s")
        print(f"  >> {r['response'][:200]}")
    return {"sender": s, "receiver": r, "tx_tokens": s["tokens"], "total_tokens": s["tokens"] + r["tokens"]}

def chain(agents, message, max_tokens=256, verbose=True):
    current = message
    results = []
    for agent in agents:
        r = agent.say(current, max_tokens=max_tokens)
        results.append({"agent": agent.name, **r})
        if verbose:
            print(f"[{agent.name}] {r['tokens']}tok, {r['time']}s")
            print(f"  >> {r['response'][:200]}")
        current = r["response"]
    return {"steps": results, "final": results[-1]["response"], "total_tokens": sum(r["tokens"] for r in results)}

def extract_number(text):
    nums = re.findall(r'-?[\d,]+\.?\d*', text.replace(',', ''))
    return float(nums[0]) if nums else -999

def grade(got, expected, tolerance=0.1):
    if expected == 0:
        return abs(got) < tolerance
    return abs(got - expected) / abs(expected) < tolerance

load_model("Qwen/Qwen3-4B")

In [ ]:
# ============================================================
# Cell 2: 도메인 전문가 에이전트 + MMLU 로더
# ============================================================
import random
from datasets import load_dataset
from itertools import permutations
import re as _re

# --- 도메인 전문가 프롬프트 ---
MATH_PROMPT = (
    "You are ONLY a mathematics domain expert. "
    "Approach every problem purely through mathematical reasoning — "
    "equations, formulas, numerical computation, and logical deduction. "
    "Do NOT use domain-specific scientific facts or programming knowledge. "
    "Reason step-by-step and put the final answer inside \\boxed{YOUR_FINAL_ANSWER}. "
    "Your final answer must be selected from A,B,C,D. "
    "For example \\boxed{A}. Do not add any other contents inside the box."
)

SCIENCE_PROMPT = (
    "You are ONLY a natural science domain expert. "
    "Approach every problem purely through scientific reasoning — "
    "physical laws, chemical properties, biological concepts, and empirical knowledge. "
    "Do NOT use abstract mathematical proofs or programming knowledge. "
    "Reason step-by-step and put the final answer inside \\boxed{YOUR_FINAL_ANSWER}. "
    "Your final answer must be selected from A,B,C,D. "
    "For example \\boxed{A}. Do not add any other contents inside the box."
)

CS_PROMPT = (
    "You are ONLY a computer science domain expert. "
    "Approach every problem purely through computational reasoning — "
    "algorithms, data structures, programming logic, and system design. "
    "Do NOT use scientific domain knowledge or pure mathematical proofs. "
    "Reason step-by-step and put the final answer inside \\boxed{YOUR_FINAL_ANSWER}. "
    "Your final answer must be selected from A,B,C,D. "
    "For example \\boxed{A}. Do not add any other contents inside the box."
)

TX_ANALYZE_PROMPT = (
    "You are ONLY a mathematics domain expert. "
    "Approach every problem purely through mathematical reasoning — "
    "equations, formulas, numerical computation, and logical deduction. "
    "Do NOT use domain-specific scientific facts or programming knowledge. "
    "Analyze this problem from your mathematical perspective. "
    "Provide your reasoning and key observations. "
    "Do NOT give a final answer or select A/B/C/D."
)

math_agent = Agent("Math", MATH_PROMPT)
science_agent = Agent("Science", SCIENCE_PROMPT)
cs_agent = Agent("CS", CS_PROMPT)

AGENTS = {"Math": math_agent, "Science": science_agent, "CS": cs_agent}
PROMPTS = {"Math": MATH_PROMPT, "Science": SCIENCE_PROMPT, "CS": CS_PROMPT}

# --- MMLU 도메인 매핑 ---
DOMAIN_MAP = {
    "Math": [
        "elementary_mathematics", "high_school_mathematics",
        "high_school_statistics", "abstract_algebra", "college_mathematics",
    ],
    "Science": [
        "high_school_physics", "high_school_chemistry",
        "high_school_biology", "conceptual_physics", "anatomy",
    ],
    "CS": [
        "high_school_computer_science", "college_computer_science",
        "computer_security", "machine_learning", "electrical_engineering",
    ],
}

# --- MMLU 로드 + 포맷팅 ---
print("Loading MMLU...")
mmlu = load_dataset("cais/mmlu", "all", split="test")
ANSWER_MAP = {0: "A", 1: "B", 2: "C", 3: "D"}

def format_mmlu(example):
    """MMLU 예제를 ABCD 문제 텍스트로 변환."""
    q = example["question"]
    choices = example["choices"]
    text = f"{q}\nA) {choices[0]}\nB) {choices[1]}\nC) {choices[2]}\nD) {choices[3]}"
    return {"text": text, "answer": ANSWER_MAP[example["answer"]], "subject": example["subject"]}

def sample_mmlu(n, domain=None, seed=42):
    """MMLU에서 n개 샘플링. domain=None이면 전체 도메인에서 균등 샘플."""
    random.seed(seed)
    if domain:
        subjects = DOMAIN_MAP[domain]
        pool = [ex for ex in mmlu if ex["subject"] in subjects]
    else:
        # 3개 도메인에서 균등 샘플
        pool = []
        for d, subjects in DOMAIN_MAP.items():
            domain_pool = [ex for ex in mmlu if ex["subject"] in subjects]
            pool.extend(random.sample(domain_pool, min(len(domain_pool), n)))
    samples = random.sample(pool, min(n, len(pool)))
    return [format_mmlu(s) for s in samples]

def ask_agent(agent, prompt, question_text, max_tokens=256):
    """에이전트에게 ABCD 문제를 풀게 하고 답 추출."""
    msg = f"{prompt}\nQuestion: {question_text}\nYour response:"
    r = agent.say(msg, max_tokens=max_tokens)
    match = _re.search(r'\\boxed\{([A-D])\}', r["response"])
    answer = match.group(1) if match else "N/A"
    return {"answer": answer, **r}

# 도메인별 문제 수 확인
for d, subjects in DOMAIN_MAP.items():
    count = sum(1 for ex in mmlu if ex["subject"] in subjects)
    print(f"  {d}: {count} questions ({', '.join(subjects[:3])}...)")
print(f"\nAgents: {list(AGENTS.keys())}")
print("Ready.")

---
# Key Idea 1: Mutual Cognitive Context Inference (MMLU 10문제)
- **Tx**: `math_agent` (analyzer only, no final answer), **Rx**: `science_agent` (gives final answer)
- MMLU 3도메인 혼합 10문제
- 4조건: no_context / oneway_tx / oneway_rx / mutual

In [ ]:
# ============================================================
# Key Idea 1: MMLU 10문제 x 4조건
# ============================================================
Q1_math = sample_mmlu(5, domain="Math", seed=42)
Q1_sci = sample_mmlu(5, domain="Science", seed=43)
Q1 = Q1_math + Q1_sci
random.shuffle(Q1)
print(f"Sampled {len(Q1)} questions (5 Math + 5 Science)")
for q in Q1[:3]:
    print(f"  [{q['subject']}] {q['text'][:60]}... ans={q['answer']}")

# 4조건 정의: Tx/Rx 메시지에 상대 정보 추가 여부
def run_key1(questions):
    conditions = ["no_context", "oneway_tx", "oneway_rx", "mutual"]
    results = {c: [] for c in conditions}

    for qi, q in enumerate(questions):
        print(f"\n--- Q{qi+1}/{len(questions)}: [{q['subject']}] {q['text'][:80]}...")
        print(f"    Expected: {q['answer']}")

        for cond in conditions:
            # Tx: analyzer mode (no final answer)
            math_agent.set_prompt(TX_ANALYZE_PROMPT)

            tx_extra = ""
            if cond in ("oneway_tx", "mutual"):
                tx_extra = "\nThe receiver is a natural science domain expert. Tailor your analysis for scientific interpretation."
            tx_msg = f"Question: {q['text']}{tx_extra}"
            tx_out = math_agent.say(tx_msg, max_tokens=128)

            # Restore
            math_agent.set_prompt(MATH_PROMPT)

            # Rx: final answer
            rx_extra = ""
            if cond in ("oneway_rx", "mutual"):
                rx_extra = "\nThe sender is a mathematics domain expert who provides structured numerical analysis."
            rx_msg = f"Previous agent's analysis:\n{tx_out['response']}{rx_extra}\n\nBased on the above analysis, answer the original question:\n{q['text']}"
            rx_out = science_agent.say(rx_msg, max_tokens=256)

            match = _re.search(r'\\boxed\{([A-D])\}', rx_out["response"])
            ans = match.group(1) if match else "N/A"

            print(f"  [{cond}] Tx:{tx_out['tokens']}tok Rx:{rx_out['tokens']}tok → {ans} {'✓' if ans == q['answer'] else '✗'}")
            if cond == "no_context":  # 첫 조건만 상세 로그
                print(f"    Tx>> {tx_out['response'][:200]}")
                print(f"    Rx>> {rx_out['response'][:200]}")

            results[cond].append({
                "correct": ans == q["answer"],
                "answer": ans, "expected": q["answer"],
                "tx_tokens": tx_out["tokens"], "rx_tokens": rx_out["tokens"],
                "total_tokens": tx_out["tokens"] + rx_out["tokens"],
            })

    return results

print("\nRunning Key Idea 1...")
k1_results = run_key1(Q1)

# 결과표
print(f"\n{'Condition':<15} {'Accuracy':<12} {'Avg Tx tok':<12} {'Avg Total':<12}")
print("-" * 50)
for cond, res in k1_results.items():
    acc = sum(r["correct"] for r in res) / len(res)
    avg_tx = sum(r["tx_tokens"] for r in res) / len(res)
    avg_tot = sum(r["total_tokens"] for r in res) / len(res)
    print(f"{cond:<15} {acc*100:>5.1f}%      {avg_tx:>8.1f}    {avg_tot:>8.1f}")

---
# Key Idea 2: Stage-Wise CoT Switching (MMLU 10문제)
- **Tx**: `math_agent` 3단계 CoT (이해→추론→전달)
- **Rx**: `science_agent` 1단계 or 3단계
- 4조건: all_general / all_aware / tx_switch / both_switch

In [ ]:
# ============================================================
# Key Idea 2: MMLU 10문제 x 4조건 (CoT Switching)
# ============================================================
Q2_math = sample_mmlu(5, domain="Math", seed=99)
Q2_sci = sample_mmlu(5, domain="Science", seed=100)
Q2 = Q2_math + Q2_sci
random.shuffle(Q2)
print(f"Sampled {len(Q2)} questions (5 Math + 5 Science)")

# Tx CoT 메시지 템플릿
TX_COT_GEN = [
    "Read this question and identify the key information.\nQuestion: {q}",
    "Based on your analysis, determine what approach is needed to solve this.\n{prev}",
    "Provide your final answer.\n{prev}",
]
TX_COT_AWR = [
    "Read this question and identify the key information.\nQuestion: {q}",
    "The receiver is a science expert. Identify the most relevant reasoning for them.\n{prev}",
    "Format your answer clearly for the science expert receiver.\n{prev}",
]
# Rx 메시지 템플릿
RX_1 = ["Previous agent's analysis:\n{prev}\n\nBased on the above, answer the original question:\n{q}"]
RX_3 = [
    "The math expert sent this analysis. Parse the key reasoning.\n{prev}",
    "Based on the parsed reasoning, determine the answer.\n{prev}",
    "Provide only the final answer.\n{prev}",
]

cot_conds = {
    "all_general":  {"tx": TX_COT_GEN, "rx": RX_1},
    "all_aware":    {"tx": TX_COT_AWR, "rx": RX_1},
    "tx_switch":    {"tx": [TX_COT_GEN[0], TX_COT_GEN[1], TX_COT_AWR[2]], "rx": RX_1},
    "both_switch":  {"tx": [TX_COT_GEN[0], TX_COT_GEN[1], TX_COT_AWR[2]], "rx": RX_3},
}

def run_key2(questions):
    results = {c: [] for c in cot_conds}

    for qi, q in enumerate(questions):
        if (qi+1) % 5 == 0:
            print(f"  Q{qi+1}/{len(questions)}...")

        for cond, cfg in cot_conds.items():
            # Tx CoT 3단계 (math_agent, system prompt = MATH_PROMPT)
            prev = ""
            tx_total = 0
            for tmpl in cfg["tx"]:
                msg = tmpl.format(q=q['text'], prev=prev)
                r = math_agent.say(msg)
                prev = r["response"]
                tx_total += r["tokens"]
            tx_msg_tok = r["tokens"]

            # Rx (science_agent, system prompt = SCIENCE_PROMPT)
            rx_total = 0
            for tmpl in cfg["rx"]:
                msg = tmpl.format(prev=prev, q=q['text'])
                r = science_agent.say(msg)
                prev = r["response"]
                rx_total += r["tokens"]

            match = _re.search(r'\\boxed\{([A-D])\}', r["response"])
            ans = match.group(1) if match else "N/A"

            results[cond].append({
                "correct": ans == q["answer"],
                "tx_msg_tokens": tx_msg_tok, "tx_cot_tokens": tx_total,
                "rx_tokens": rx_total,
                "total_tokens": tx_total + rx_total,
            })
    return results

print("\nRunning Key Idea 2...")
k2_results = run_key2(Q2)

print(f"\n{'Condition':<15} {'Accuracy':<10} {'Avg TxMsg':<10} {'Avg CoT':<10} {'Avg Total':<10}")
print("-" * 55)
for cond, res in k2_results.items():
    acc = sum(r["correct"] for r in res) / len(res)
    avg_msg = sum(r["tx_msg_tokens"] for r in res) / len(res)
    avg_cot = sum(r["tx_cot_tokens"] for r in res) / len(res)
    avg_tot = sum(r["total_tokens"] for r in res) / len(res)
    print(f"{cond:<15} {acc*100:>5.1f}%    {avg_msg:>7.1f}   {avg_cot:>7.1f}   {avg_tot:>7.1f}")

---
# Key Idea 4: Task Scheduling (MMLU 30문제, 도메인별 10개)
- **3 에이전트**: `Math`, `Science`, `CS` (고정)
- A→B→C 체인, 6순열 전부 테스트
- 도메인별 10문제 x 3도메인 = 30문제
- 도메인 매칭 순서가 정답률에 미치는 영향 측정

In [ ]:
# ============================================================
# Key Idea 4: MMLU 30문제 x 6순열
# ============================================================
Q4_math = sample_mmlu(10, domain="Math", seed=77)
Q4_sci  = sample_mmlu(10, domain="Science", seed=78)
Q4_cs   = sample_mmlu(10, domain="CS", seed=79)
Q4_all  = [(q, "Math") for q in Q4_math] + [(q, "Science") for q in Q4_sci] + [(q, "CS") for q in Q4_cs]
print(f"Sampled: Math={len(Q4_math)}, Science={len(Q4_sci)}, CS={len(Q4_cs)}, Total={len(Q4_all)}")

agent_names = ["Math", "Science", "CS"]
all_orders = list(permutations(agent_names))

def run_chain_mmlu(order, question_text):
    """3-agent chain, 각 에이전트는 자기 도메인 프롬프트로 처리."""
    prev = f"Question: {question_text}"
    total_tokens = 0
    for i, name in enumerate(order):
        if i == 0:
            msg = prev
        else:
            msg = f"Previous agent's analysis:\n{prev}\n\nAnswer the original question:\n{question_text}"
        r = AGENTS[name].say(msg)
        prev = r["response"]
        total_tokens += r["tokens"]
    match = _re.search(r'\\boxed\{([A-D])\}', prev)
    return match.group(1) if match else "N/A", total_tokens

# 실행
k4_results = {}
for order in all_orders:
    order_key = "→".join(order)
    domain_scores = {"Math": [], "Science": [], "CS": []}
    total_tok = 0

    for q, domain in Q4_all:
        ans, tok = run_chain_mmlu(order, q["text"])
        domain_scores[domain].append(ans == q["answer"])
        total_tok += tok

    all_scores = [s for scores in domain_scores.values() for s in scores]
    k4_results[order_key] = {
        "accuracy": sum(all_scores) / len(all_scores),
        "by_domain": {d: sum(s)/len(s) for d, s in domain_scores.items()},
        "correct": sum(all_scores),
        "total": len(all_scores),
        "total_tokens": total_tok,
    }
    print(f"  {order_key}: {sum(all_scores)}/{len(all_scores)} "
          f"(M:{sum(domain_scores['Math'])}/10 S:{sum(domain_scores['Science'])}/10 C:{sum(domain_scores['CS'])}/10)")

print("\nDone.")

In [ ]:
# ============================================================
# Key Idea 4: 결과 비교표
# ============================================================
sorted_k4 = sorted(k4_results.items(), key=lambda x: -x[1]["accuracy"])

print(f"{'Order':<20} {'Total':<10} {'Math':<8} {'Science':<8} {'CS':<8} {'Tokens'}")
print("-" * 65)
for order_key, r in sorted_k4:
    d = r["by_domain"]
    print(f"{order_key:<20} {r['accuracy']*100:>5.1f}%    "
          f"{d['Math']*100:>5.1f}%  {d['Science']*100:>5.1f}%  {d['CS']*100:>5.1f}%  {r['total_tokens']}")

print(f"\nBest:  {sorted_k4[0][0]} ({sorted_k4[0][1]['accuracy']*100:.1f}%)")
print(f"Worst: {sorted_k4[-1][0]} ({sorted_k4[-1][1]['accuracy']*100:.1f}%)")

# 도메인별 최적 순서
print("\nDomain-optimal order (last agent = domain expert):")
for domain in ["Math", "Science", "CS"]:
    best_order = max(k4_results.items(), key=lambda x: x[1]["by_domain"][domain])
    print(f"  {domain}: {best_order[0]} ({best_order[1]['by_domain'][domain]*100:.0f}%)")